# 14 — 2022 stress test

Measure how exceptional 2022 was relative to pre-closure distributions and decompose which physical-balance components moved.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths
from portugal_refining_resilience.io import persist_dataframe, write_json
from portugal_refining_resilience.metrics import benchmark_deviation

PATHS = get_paths(ROOT)
pd.set_option("display.max_columns", 100)


In [ ]:
panel = pd.read_csv(PATHS.processed / "fuel_annual_analytical_panel.csv")
records = []
baseline_windows = [(2013, 2019), (2013, 2020), (2015, 2020)]
outcomes = ["exports_kt", "imports_kt", "demand_kt", "refinery_output_kt", "net_imports_kt", "net_import_to_demand_ratio"]
for outcome in outcomes:
    if outcome not in panel.columns:
        continue
    for baseline_start, baseline_end in baseline_windows:
        part = benchmark_deviation(panel, value_column=outcome, target_year=2022, baseline_start=baseline_start, baseline_end=baseline_end)
        records.append(part)
stress = pd.concat(records, ignore_index=True) if records else pd.DataFrame()
persist_dataframe(stress, PATHS.metrics / "stress_2022_metrics.csv")
display(stress.sort_values(["value_column", "baseline_start", "product"]))


In [ ]:
# Direct year-on-year movement around the stress year.
window = panel.loc[panel["year"].isin([2020, 2021, 2022, 2023])].copy()
cols = [c for c in ["exports_kt", "imports_kt", "demand_kt", "refinery_output_kt", "net_imports_kt"] if c in window.columns]
yoy_rows = []
for product, sub in window.groupby("product"):
    sub = sub.sort_values("year")
    for col in cols:
        values = sub.set_index("year")[col]
        changes = values.diff()
        lag = values.shift(1)
        pct_changes = values.pct_change(fill_method=None) * 100 if (lag.dropna() > 0).all() and (values >= 0).all() else None
        for year, value in changes.dropna().items():
            row = {"product": product, "outcome": col, "year": int(year), "yoy_change": value}
            if pct_changes is not None:
                row["yoy_pct"] = pct_changes.loc[year]
            yoy_rows.append(row)
yoy = pd.DataFrame(yoy_rows)
persist_dataframe(yoy, PATHS.metrics / "stress_window_yoy_changes.csv")
display(yoy.loc[yoy["year"] == 2022])


In [ ]:
# Source sensitivity. Some 2019-2024 trade cells are flagged for review in the
# JODI/DGEG reconciliation, and 2022 diesel exports is the largest single gap.
# A headline stress statistic must not rest on the series the reconciliation
# identifies as the outlier, so recompute it on the corroborated value too.
recon_path = PATHS.metrics / "jodi_dgeg_trade_reconciliation.csv"
sens_rows = []
if recon_path.exists():
    recon = pd.read_csv(recon_path)
    flagged = recon.loc[recon["reconciliation_status"].ne("within_tolerance")]
    for row in flagged.itertuples():
        column = f"{row.flow}_kt"
        subset = panel.loc[panel["product"].eq(row.product)]
        base = subset.loc[subset["year"].between(2013, 2019), column].dropna()
        if base.empty or len(base) < 2:
            continue
        for label, value in (("JODI", row.value_kt_jodi), ("DGEG", row.value_kt_dgeg)):
            sens_rows.append({
                "year": int(row.year), "product": row.product, "flow": row.flow,
                "source": label, "value_kt": float(value),
                "baseline_mean": float(base.mean()),
                "baseline_std": float(base.std(ddof=1)),
                "z_score": float((value - base.mean()) / base.std(ddof=1)),
                "empirical_percentile": float(100 * (base <= value).mean()),
                "baseline_start": 2013, "baseline_end": 2019,
            })
stress_source_sensitivity = pd.DataFrame(sens_rows)
if not stress_source_sensitivity.empty:
    persist_dataframe(
        stress_source_sensitivity,
        PATHS.metrics / "stress_2022_source_sensitivity.csv",
        key_columns=["year", "product", "flow", "source"],
    )
    display(stress_source_sensitivity)
else:
    print("No flagged reconciliation cells; source sensitivity not required.")